# TN-VQE on IBM hardware: generating and costing the campaign

This notebook builds the stage-1 screening campaign, costs every run against
measured IBM billing, and partitions the result into batches sized to IBM's
access plans. It is the executable counterpart to
[the campaign README](README.md),
which carries the experimental reasoning behind the choices made here.

A **run** is one VQE or TN-VQE optimisation of one Hamiltonian by one
method, with its own ansatz, mapper, measurement method and evaluation
budget. One run is one row of the generated CSV.

Everything below needs only the standard library and this repository. No
IBM credentials are used, and nothing is submitted to hardware.

In [ ]:
import pathlib
import sys

REPO = pathlib.Path.cwd()
while not (REPO / "pyproject.toml").exists():
    REPO = REPO.parent

sys.path.insert(0, str(REPO / "src"))
sys.path.insert(0, str(REPO / "utils"))

import build_benchmark_matrix as matrix
import split_benchmark_batches as batches

CAMPAIGN = REPO / "data" / "benchmarks" / "ibm_tn-vqe_qesem"
print(f"repository: {REPO}")

## Stage 1: the screening campaign

Stage 1 crosses the discrete experimental factors: 2 molecules, 6 basis
sets, 2 fermion-to-qubit mappers, 2 ansatz types and 3 methods. The
crossing is not complete, and the gaps are decisions rather than
omissions: pairs listed in `SKIPPED_PAIRS`, Jordan-Wigner runs above
`MAX_JW_QUBITS`, and the `pauli` half of the pairs in `GROUPED_ONLY`.

In [ ]:
rows = matrix.build_stage1()

# write_csv() assigns Case_ID as it writes; do the same here so the costing
# below can key on it without touching the committed files.
for i, row in enumerate(rows, start=1):
    row["Case_ID"] = str(i)

print(f"{len(rows)} runs")
print(f"  bases:     {', '.join(matrix.BASES)}")
print(f"  ansatz:    {', '.join(matrix.ANSATZE)}")
print(f"  geometries: {matrix.GEOMETRIES}")

Each of the three methods runs every ansatz type on the same Hamiltonian,
from the same pinned OpenQASM file and at the same initial parameters, so
that a difference between them is attributable to the method alone. The
`network` method takes no quantum measurements at all: it optimises the
tensor network classically at frozen circuit parameters, and is the
baseline the hardware results are read against.

In [ ]:
import collections

by_method = collections.Counter(r["Method"] + " / " + r["Optimization_Mode"] for r in rows)
for key, count in sorted(by_method.items()):
    print(f"  {key:28} {count:>4} runs")

## What one evaluation costs

Evaluating the expectation value ⟨H⟩ requires one circuit per measurement
basis rather than a single circuit. Denoting that count `E`, it is a
property of the Hamiltonian rather than of the circuit preparing the
state, since it follows from the number of mutually commuting sets into
which the Hamiltonian's Pauli terms partition.

The cost line below is fitted to completed `ibm_aachen` jobs whose billed
`quantum_seconds` are known, at the shot count and options this campaign
submits under. It holds to within 4% from `E = 2` to `E = 81`, which
brackets the 2 to 37 this campaign occupies.

In [ ]:
print(f"billed seconds per evaluation = {batches._FIXED_S_PER_EVALUATION}"
      f" + {batches._S_PER_MEASUREMENT_BASIS} x E\n")
for e in (2, 5, 16, 29, 37):
    print(f"  E = {e:>2}   {batches.evaluation_seconds(e):>5.1f} s per evaluation")

A run's total is that figure times its own evaluation budget, `Iterations`,
which is proportional to its free-parameter count rather than a flat value:
`max(30, ceil(1.3 n))`. COBYLA needs evaluations in proportion to the
parameter count to make a given amount of progress, so a flat budget would
reach a shrinking fraction of the achievable descent as circuits widen,
making the optimizer budget a confound correlated with qubit count.

In [ ]:
per_run = batches.estimate_per_row_qpu_seconds(rows)
total_s = sum(per_run.values())
evaluations = sum(int(r["Iterations"]) for r in rows if int(r["Case_ID"]) in per_run)

print(f"{len(per_run)} costed runs, {len(rows) - len(per_run)} classical-only")
print(f"{evaluations:,} cost-function evaluations")
print(f"{total_s / 60:,.0f} minutes of QPU time")

### Where the time goes

Cost is driven by the Hamiltonian's measurement count, not by circuit
depth, so grouping by mapper, molecule and qubit count accounts for the
whole budget. Runs marked `assumed` carry the largest value measured on
that mapper because their Hamiltonians cannot be built offline: those are
lower bounds, and they are the reason this allocation is not yet a
purchase plan.

In [ ]:
spend = collections.defaultdict(lambda: [0, 0.0])
for r in rows:
    cid = int(r["Case_ID"])
    if cid not in per_run:
        continue
    key = (r["Mapper"], r["Molecule"], int(r["N_Qubit"]),
           int(r["Num_ExpVals_Per_Iter"]), r["Num_ExpVals_Source"])
    spend[key][0] += int(r["Iterations"])
    spend[key][1] += per_run[cid]

print(f"{'mapper':9}{'mol':5}{'q':>3}{'E':>4}  {'source':14}{'evals':>7}{'s/eval':>8}{'min':>8}{'share':>7}")
for (mapper, mol, q, e, source), (evals, secs) in sorted(spend.items(), key=lambda kv: -kv[1][1]):
    print(f"{mapper:9}{mol:5}{q:>3}{e:>4}  {source:14}{evals:>7,}"
          f"{batches.evaluation_seconds(e):>8.1f}{secs / 60:>8,.0f}{100 * secs / total_s:>6.0f}%")

assumed = sum(s for (_, _, _, _, src), (_, s) in spend.items() if src == "assumed")
print(f"\n{100 * assumed / total_s:.0f}% of the estimate rests on assumed measurement counts")

## Allocation to IBM access plans

Each plan is a separate purchase, so the batches are not one running total:
the Open Plan's free 10 minutes are not deducted from a Flex purchase, and
neither is deducted from a Premium allocation. Runs are sorted by ascending
cost and each batch accepts runs until the next would exceed its budget,
which makes batch1 the cheapest work available and therefore a pipeline
validation run before any time is bought.

In [ ]:
cut, overflow, unestimable = batches.split_into_batches(rows, per_run)
classical_only = [r for r in rows if batches.is_classical_only(r)]

print(f"{'file':32}{'runs':>6}{'minutes':>10}{'budget':>10}")
print(f"{'batch0_classical_only.csv':32}{len(classical_only):>6}{0.0:>10.2f}{'none':>10}")
for (name, budget_s), batch in zip(batches._PLAN_BUDGETS_S, cut):
    spent = sum(per_run[int(r['Case_ID'])] for r in batch)
    print(f"{name + '.csv':32}{len(batch):>6}{spent / 60:>10.2f}{budget_s // 60:>10}")

print(f"\noverflow: {len(overflow)} runs, uncostable: {len(unestimable)} runs")

## Writing the campaign to disk

The two guides below are what actually generate the committed files. The
partition is regenerated rather than edited, since any change to the shot
count, the evaluation budget, the ansatz set or a measurement count moves
the batch boundaries.

In [ ]:
# Uncomment to overwrite the committed campaign files.
# matrix.write_csv(CAMPAIGN / "stage1_screening_matrix.csv", rows)
# batches.main()

print("stage 1:")
print("  PYTHONPATH=src python utils/build_benchmark_matrix.py")
print("  PYTHONPATH=src python utils/split_benchmark_batches.py")

## Later stages

Stages 2 and 3 are generated on demand, since the inputs of each are an
output of the stage before it. Neither will generate without an explicit
selection: no `--select`, no `--refine`, no `--precision`, no output. A
silently defaulted selection would make the provenance of a later stage
unrecoverable.

Stage 2 sweeps the tensor-network parameters on the combinations stage 1
selected, at a larger evaluation budget (`4n`, for about 80% of achievable
descent against stage 1's roughly 50%):

```sh
PYTHONPATH=src python utils/build_benchmark_matrix.py --stage 2 \
    --select H2=cc-pvdz --select H2O=def2-tzvp --ansatz EfficientSU2_circular
```

Stage 3 resubmits converged stage-2 parameters once, mitigated and
unmitigated, through Qedma's QESEM service:

```sh
PYTHONPATH=src python utils/build_benchmark_matrix.py --stage 3 \
    --from data/benchmarks/ibm_tn-vqe_qesem/stage2_deep_sweep.csv \
    --refine 17=results/converged/case_17.json --precision 0.0016
```